In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, T5ForConditionalGeneration, TrainingArguments

In [2]:
train_data  = pd.read_csv("samsum-train.csv")
val_data  = pd.read_csv("samsum-validation.csv")

In [3]:
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data  = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [4]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # lines
    text = re.sub(r"\s+", " ", text) # spaces
    text = re.sub(r"<.*?>", " ", text) # html tags <p>, <h1>
    text = text.strip().lower()
    return text
    

In [5]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [6]:
# Tokenize data
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [7]:
def tokenize(data): 
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"] # token ids add to labels
    return inputs

In [8]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()
# input ids =  dialogue => token_ids
# 1=> End of Sequence, => added padding
# attention mask
# labels - target => summary token

In [9]:
print(len(train_dataset[0]["input_ids"]))
print(len(train_dataset[0]["labels"]))

# as set before.

512
150


In [10]:
# Working with model
# NLP -> Generation Task
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [11]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")

print("device: ", device)
model.to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [12]:
# Training arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs=10 ,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.635412,0.382887
2,0.397168,0.359311
3,0.373466,0.353370
4,0.359934,0.348837
5,0.350766,0.347088
6,0.343895,0.344926
7,0.339062,0.343833
8,0.334702,0.343937
9,0.331143,0.343742
10,0.330573,0.343844


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5000, training_loss=0.6796121612548828, metrics={'train_runtime': 8058.7165, 'train_samples_per_second': 4.964, 'train_steps_per_second': 0.62, 'total_flos': 5413672058880000.0, 'train_loss': 0.6796121612548828, 'epoch': 10.0})

In [15]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [16]:
model = T5ForConditionalGeneration. from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
# Testing core logic for summarization
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # cleaned and pre processed
    # Tokenizer
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length = 512,
        truncation = True,
        return_tensors="pt"
    ).to(device)
    
    # Generate the Summary
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )
    # Token ids convert to summary => descending
    summary = tokenizer.decode(targets[0], skip_special_tokens=True)
    return summary

In [18]:
test_dialogue = """Alex: Hey, did you finish that book I lent you?Sam: Almost! I have about twenty pages left. It’s getting intense.Alex: I told you! I couldn't put it down during the final chapter.Sam: No spoilers, please. I’m planning to finish it over coffee tonight.Alex: Fair enough. Just text me when you’re done so we can talk about that ending.Sam: Will do. It better be as good as you promised!"""

summary = summarize_dialogue(test_dialogue)
print("Summary: ", summary)

Summary:  alex has about twenty pages left. he's planning to finish the book over coffee tonight.
